# 📧 Email Reply Draft Assistant

## What We're Building

An assistant that reads an incoming email and:
1. **Classifies** the email intent (complaint, inquiry, meeting request, etc.)
2. **Suggests** an appropriate reply tone (formal, friendly, empathetic)
3. **Generates 3 different reply options** so you can pick the best one

## How It Works

```
Input Email → Classify Intent → Determine Tone → Generate 3 Reply Drafts
                  ↓                    ↓                    ↓
           (complaint,            (empathetic,        (short, medium,
            inquiry,              formal,              detailed)
            request...)           friendly...)
```

## Key LangChain Concept: Structured Output

Instead of getting free-form text from the LLM, we use **Pydantic models** to
define the exact shape of the output. LangChain will ensure the LLM returns
data matching our schema — with proper types, required fields, and validation.

## Stack
- **LangChain** — orchestration with structured output
- **Pydantic** — data validation and schema definition
- **Ollama (qwen3.5:9b)** — local LLM

## Step 1 — Install & Import Dependencies

In [ ]:
# Install if needed (uncomment)
# !pip install langchain langchain-ollama -q

from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field
from enum import Enum
import json

print("All imports successful!")

## Step 2 — Define Output Schemas with Pydantic

Pydantic models describe the **exact structure** we want from the LLM.
This is much more reliable than hoping the model returns parseable text.

```python
# Instead of: "Please return JSON with intent and confidence..."
# We define:  A Pydantic model → LangChain enforces the schema
```

In [ ]:
class EmailClassification(BaseModel):
    """Classification of an incoming email."""
    intent: str = Field(
        description="The primary intent: complaint, inquiry, meeting_request, "
                    "follow_up, feedback, introduction, or other"
    )
    urgency: str = Field(
        description="Urgency level: low, medium, or high"
    )
    suggested_tone: str = Field(
        description="Suggested reply tone: formal, friendly, empathetic, or professional"
    )
    key_points: list[str] = Field(
        description="The main points or questions that need to be addressed"
    )
    summary: str = Field(
        description="One-sentence summary of the email"
    )


class ReplyDraft(BaseModel):
    """A single reply draft."""
    style: str = Field(description="Style label: concise, balanced, or detailed")
    subject: str = Field(description="Reply email subject line")
    body: str = Field(description="The full reply email body")


class ReplyOptions(BaseModel):
    """Three reply drafts for the user to choose from."""
    replies: list[ReplyDraft] = Field(
        description="Exactly 3 reply drafts: concise, balanced, and detailed"
    )


print("Schemas defined!")
print(f"\nEmailClassification fields: {list(EmailClassification.model_fields.keys())}")
print(f"ReplyDraft fields: {list(ReplyDraft.model_fields.keys())}")

## Step 3 — Initialize the LLM

In [ ]:
llm = ChatOllama(
    model="qwen3.5:9b",
    temperature=0.4,
)

print("LLM ready!")

## Step 4 — Build the Classification Chain

We use `JsonOutputParser` with our Pydantic model to get structured output.
The parser automatically adds format instructions to the prompt.

In [ ]:
# Create a JSON output parser with our schema
classification_parser = JsonOutputParser(pydantic_object=EmailClassification)

classification_prompt = ChatPromptTemplate.from_template(
    """You are an email analysis assistant. Classify the following email.

{format_instructions}

Email:
---
{email}
---

Provide your classification as JSON:"""
)

# Build the chain
classify_chain = classification_prompt | llm | classification_parser

# Show what format instructions look like
print("Format instructions sent to LLM:")
print(classification_parser.get_format_instructions()[:300])

## Step 5 — Test Classification with Sample Emails

In [ ]:
# Sample emails for testing
sample_emails = {
    "complaint": """Subject: Extremely disappointed with recent order #4582

Hi,

I ordered the Premium Wireless Headphones (Order #4582) two weeks ago and they 
arrived with a cracked case and the left ear cup doesn't produce any sound. 
This is the second defective product I've received from your store.

I need an immediate replacement or a full refund. I've been a loyal customer 
for 3 years and this experience has been really frustrating.

Please respond ASAP.

Thanks,
Sarah Johnson""",

    "meeting_request": """Subject: Q3 Planning Session - Can we meet Thursday?

Hi team,

I'd like to schedule a Q3 planning session to discuss our roadmap priorities 
and resource allocation. Would Thursday at 2 PM work for everyone?

Agenda:
1. Review Q2 metrics
2. Discuss Q3 OKRs  
3. Resource planning

Please confirm your availability.

Best,
Mike Chen""",

    "inquiry": """Subject: Question about Enterprise API pricing

Hello,

We're evaluating your API platform for our company (500+ employees). 
Could you provide information on:
- Enterprise pricing tiers
- SLA guarantees
- Custom integration support

We'd also like to know if you offer a pilot program.

Regards,
David Park
CTO, TechCorp"""
}

# Classify the complaint email
email_text = sample_emails["complaint"]
classification = classify_chain.invoke({
    "email": email_text,
    "format_instructions": classification_parser.get_format_instructions(),
})

print("📧 Classification Result:")
print(json.dumps(classification, indent=2))

In [ ]:
# Classify the meeting request
classification_meeting = classify_chain.invoke({
    "email": sample_emails["meeting_request"],
    "format_instructions": classification_parser.get_format_instructions(),
})

print("📧 Meeting Email Classification:")
print(json.dumps(classification_meeting, indent=2))

## Step 6 — Build the Reply Generator

Using the classification results, we generate **3 reply options**:
1. **Concise** — short and to the point
2. **Balanced** — professional with adequate detail
3. **Detailed** — thorough and comprehensive

In [ ]:
reply_parser = JsonOutputParser(pydantic_object=ReplyOptions)

reply_prompt = ChatPromptTemplate.from_template(
    """You are a professional email writer. Based on the email classification below,
generate exactly 3 reply drafts.

Email Classification:
- Intent: {intent}
- Urgency: {urgency}
- Suggested Tone: {tone}
- Key Points to Address: {key_points}

Original Email:
---
{email}
---

Generate 3 replies:
1. CONCISE - Brief, 2-3 sentences
2. BALANCED - Professional, medium length
3. DETAILED - Thorough, addresses every point

All replies should match the suggested tone ({tone}).

{format_instructions}

Provide the 3 replies as JSON:"""
)

reply_chain = reply_prompt | llm | reply_parser

print("Reply generator chain ready!")

In [ ]:
# Generate replies for the complaint email
key_points_str = ", ".join(classification.get("key_points", []))

replies = reply_chain.invoke({
    "intent": classification.get("intent", "complaint"),
    "urgency": classification.get("urgency", "high"),
    "tone": classification.get("suggested_tone", "empathetic"),
    "key_points": key_points_str,
    "email": email_text,
    "format_instructions": reply_parser.get_format_instructions(),
})

# Display the replies
for reply in replies.get("replies", []):
    print(f"\n{'='*60}")
    print(f"📝 Style: {reply['style'].upper()}")
    print(f"📋 Subject: {reply['subject']}")
    print(f"{'-'*60}")
    print(reply['body'])

## Step 7 — Complete Pipeline Function

Let's wrap everything into a single function for easy use.

In [ ]:
def process_email(email_text: str) -> dict:
    """Complete pipeline: classify email → generate 3 reply options."""
    
    # Step 1: Classify
    print("🔍 Classifying email...")
    classification = classify_chain.invoke({
        "email": email_text,
        "format_instructions": classification_parser.get_format_instructions(),
    })
    
    print(f"  Intent: {classification.get('intent')}")
    print(f"  Urgency: {classification.get('urgency')}")
    print(f"  Tone: {classification.get('suggested_tone')}")
    print(f"  Summary: {classification.get('summary')}")
    
    # Step 2: Generate replies
    print("\n✍️ Generating reply drafts...")
    key_points_str = ", ".join(classification.get("key_points", []))
    
    replies = reply_chain.invoke({
        "intent": classification.get("intent", "unknown"),
        "urgency": classification.get("urgency", "medium"),
        "tone": classification.get("suggested_tone", "professional"),
        "key_points": key_points_str,
        "email": email_text,
        "format_instructions": reply_parser.get_format_instructions(),
    })
    
    # Display
    for i, reply in enumerate(replies.get("replies", []), 1):
        print(f"\n{'='*60}")
        print(f"Option {i}: {reply['style'].upper()}")
        print(f"Subject: {reply['subject']}")
        print(f"{'-'*60}")
        print(reply['body'])
    
    return {"classification": classification, "replies": replies}


# Try with the inquiry email
result = process_email(sample_emails["inquiry"])

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Pydantic Models** | Define exact output schema with types and descriptions |
| **JsonOutputParser** | Forces LLM output to match our Pydantic schema |
| **Format Instructions** | Auto-generated text telling the LLM how to format output |
| **Chain Composition** | `prompt \| llm \| parser` — LCEL for clean pipelines |
| **Two-Stage Pipeline** | Classify first, then use classification to guide generation |

## 🔧 Customization Ideas

- **Add more intents**: "job application", "newsletter", "legal notice"
- **Tone presets**: Let users override the suggested tone
- **Context injection**: Add company name, sender role, or previous thread
- **Multi-language**: Add `reply_language` field to generate replies in other languages